## Introduccion al Scoring

El sistema de scoring asigna una puntuacion de confianza (0.0 - 1.0)
a cada patron detectado, permitiendo:

- Rankear patrones por relevancia
- Identificar el patron principal
- Filtrar falsos positivos
- Resolver conflictos entre patrones

### Niveles de Confianza

| Nivel | Rango | Interpretacion |
|-------|-------|----------------|
| Muy Alto | >= 0.90 | Deteccion casi segura |
| Alto | 0.70 - 0.89 | Deteccion confiable |
| Medio | 0.50 - 0.69 | Deteccion probable |
| Bajo | 0.30 - 0.49 | Deteccion posible |
| Muy Bajo | < 0.30 | Deteccion dudosa |

### Enum de Niveles

```python
class ConfidenceLevel(str, Enum):
    VERY_LOW = "very_low"      # < 0.3
    LOW = "low"                # 0.3 - 0.5
    MEDIUM = "medium"          # 0.5 - 0.7
    HIGH = "high"              # 0.7 - 0.9
    VERY_HIGH = "very_high"    # >= 0.9
```

## Componentes del Score

El score final se calcula considerando multiples factores:

### 1. Score Base (Indicadores)

Calculado a partir de los indicadores encontrados:

```
score_base = sum(peso_indicador * encontrado) / sum(peso_indicador)
```

Ejemplo:

| Indicador | Peso | Encontrado | Contribucion |
|-----------|------|------------|---------------|
| llamada_recursiva | 2.0 | Si | 2.0 |
| calculo_medio | 1.5 | Si | 1.5 |
| fase_combinacion | 1.0 | No | 0.0 |
| caso_base | 1.0 | Si | 1.0 |

```
score_base = (2.0 + 1.5 + 1.0) / (2.0 + 1.5 + 1.0 + 1.0) = 4.5 / 5.5 = 0.818
```

### 2. Ajustes por Contexto

| Factor | Efecto | Valor |
|--------|--------|-------|
| Indicadores faltantes | Penalizacion | -0.10 |
| Patrones conflictivos | Penalizacion | -0.15 |
| Alta confianza inicial | Bonus | +10% |

### 3. Normalizacion

El score final se normaliza al rango [0.0, 1.0]:

```python
score_final = max(0.0, min(1.0, score_ajustado))
```

## Pesos y Umbrales

### Pesos de Componentes

El PatternScorer utiliza los siguientes pesos:

```python
self.weights = {
    "ast_structure": 0.35,     # Estructura del AST
    "code_keywords": 0.25,     # Palabras clave detectadas
    "complexity_match": 0.20,  # Coincidencia con complejidad tipica
    "confidence": 0.20         # Confianza del detector
}
```

### Penalizaciones

```python
self.penalties = {
    "conflicting_patterns": -0.15,  # Patron en conflicto detectado
    "missing_indicators": -0.10     # >50% indicadores faltantes
}
```

### Umbrales de Confianza

```python
self.thresholds = {
    "high_confidence": 0.80,   # Umbral para bonus
    "medium_confidence": 0.60, # Umbral medio
    "low_confidence": 0.40     # Umbral minimo para reportar
}
```

### Umbral por Defecto

El PatternDetector usa un umbral minimo de **0.30** para reportar un patron.
Patrones con confianza menor son descartados.

## Resolucion de Conflictos

Algunos patrones son mutuamente excluyentes o poco probables de coexistir.

### Matriz de Exclusion

```python
self.mutually_exclusive = {
    PatternType.BRUTE_FORCE: [
        PatternType.DYNAMIC_PROGRAMMING,
        PatternType.GREEDY
    ],
    PatternType.DYNAMIC_PROGRAMMING: [
        PatternType.BRUTE_FORCE
    ],
    PatternType.GREEDY: [
        PatternType.BRUTE_FORCE,
        PatternType.BACKTRACKING
    ]
}
```

### Logica de Conflicto

```
Si patron_A detectado con confianza 0.75
Y patron_B detectado con confianza 0.65
Y patron_A excluye patron_B:

    patron_A.score = 0.75 (sin cambio)
    patron_B.score = 0.65 - 0.15 = 0.50 (penalizado)
```

### Ejemplo: Fuerza Bruta vs DP

Un algoritmo puede mostrar indicadores de ambos:

| Patron | Score Base | Conflicto | Score Final |
|--------|------------|-----------|-------------|
| Fuerza Bruta | 0.55 | DP detectado | 0.40 |
| Prog. Dinamica | 0.80 | FB detectado | 0.80 |

DP gana porque tiene mayor confianza inicial.

## Calibracion de Parametros

### Proceso de Calibracion

Los parametros del scorer se calibran mediante:

1. **Dataset de referencia**: Algoritmos conocidos con patrones etiquetados
2. **Metricas de evaluacion**: Precision, recall, F1-score
3. **Optimizacion**: Ajuste de pesos para maximizar metricas

### Metricas Objetivo

| Metrica | Objetivo | Descripcion |
|---------|----------|-------------|
| Precision | > 0.85 | Patrones correctos / detectados |
| Recall | > 0.80 | Patrones detectados / existentes |
| F1-Score | > 0.82 | Media armonica |

### Ajuste de Pesos por Patron

Cada detector puede tener pesos personalizados para sus indicadores:

**Divide y Venceras**:

| Indicador | Peso Inicial | Peso Calibrado |
|-----------|--------------|----------------|
| calculo_medio | 1.0 | 1.5 |
| multiples_llamadas | 1.0 | 2.0 |
| fase_combinacion | 1.0 | 1.0 |

El indicador `multiples_llamadas` recibe mayor peso porque es
el mas distintivo del patron.

### Tabla de Referencia: Algoritmos de Prueba

| Algoritmo | Patron Esperado | Complejidad |
|-----------|-----------------|-------------|
| Bubble Sort | Ordenamiento, Fuerza Bruta | O(n^2) |
| Merge Sort | Divide y Venceras | O(n log n) |
| Fibonacci DP | Programacion Dinamica | O(n) |
| N-Queens | Backtracking | O(n!) |
| Dijkstra | Voraz | O(E log V) |
| Binary Search | Busqueda, Divide y Venceras | O(log n) |

## Estructura del ScoredPattern

El resultado del scoring es un objeto `ScoredPattern`:

```python
@dataclass
class ScoredPattern:
    pattern: PatternMatch    # Patron original detectado
    final_score: float       # Score ajustado (0.0 - 1.0)
    rank: int = 0            # Posicion en ranking (1 = primero)
    is_primary: bool = False # True si es patron principal
    conflicts: List[str] = None  # Patrones en conflicto
```

### Ejemplo de Resultado

Para un algoritmo Merge Sort:

```python
[
    ScoredPattern(
        pattern=PatternMatch(
            pattern_type=PatternType.DIVIDE_AND_CONQUER,
            pattern_name="Divide y Venceras",
            confidence=0.92
        ),
        final_score=0.92,
        rank=1,
        is_primary=True,
        conflicts=[]
    ),
    ScoredPattern(
        pattern=PatternMatch(
            pattern_type=PatternType.RECURSIVE,
            pattern_name="Recursivo",
            confidence=0.85
        ),
        final_score=0.85,
        rank=2,
        is_primary=False,
        conflicts=[]
    )
]
```

## Metodos del PatternScorer

### score_patterns(patterns)

Asigna scores a todos los patrones y los rankea:

```python
def score_patterns(self, patterns: List[PatternMatch]) -> List[ScoredPattern]:
    # 1. Calcular score base para cada patron
    # 2. Aplicar ajustes por contexto
    # 3. Detectar conflictos
    # 4. Rankear por score
    # 5. Marcar patron primario
    return scored_patterns
```

### filter_by_confidence(patterns, threshold)

Filtra patrones por umbral minimo:

```python
def filter_by_confidence(
    self,
    patterns: List[ScoredPattern],
    threshold: float
) -> List[ScoredPattern]:
    return [p for p in patterns if p.final_score >= threshold]
```

### get_primary_pattern(patterns)

Retorna el patron con mayor score:

```python
def get_primary_pattern(
    self,
    patterns: List[ScoredPattern]
) -> Optional[ScoredPattern]:
    if not patterns:
        return None
    return max(patterns, key=lambda p: p.final_score)
```

## Demostracion: Sistema de Scoring

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser.pseudocode_parser import PseudocodeParser
from app.core.patterns.pattern_detector import PatternDetector

parser = PseudocodeParser()
detector = PatternDetector()

# Ejemplo: Quick Sort (Divide y Venceras + Ordenamiento)
codigo = '''
algorithm quickSort(A[], low, high)
begin
    if (low < high) then
        p <- call partition(A, low, high)
        call quickSort(A, low, p - 1)
        call quickSort(A, p + 1, high)
    end
end
'''

ast = parser.parse(codigo)
resultado = detector.detect(ast)

print("=== Resultados del Scoring ===")
print(f"Patrones detectados: {resultado.pattern_count}\n")

for i, sp in enumerate(resultado.all_patterns, 1):
    print(f"{i}. {sp.pattern.pattern_name}")
    print(f"   Score: {sp.final_score:.2%}")
    print(f"   Nivel: {sp.pattern.confidence_level.value}")
    print(f"   Primario: {'Si' if sp.is_primary else 'No'}")
    if sp.conflicts:
        print(f"   Conflictos: {sp.conflicts}")
    print()

---

## Conclusiones

El sistema de scoring proporciona:

1. **Evaluacion objetiva**: Scores basados en indicadores verificables
2. **Manejo de ambiguedad**: Multiples patrones pueden ser detectados
3. **Resolucion de conflictos**: Penalizacion de patrones mutuamente excluyentes
4. **Calibracion**: Parametros ajustables para mejorar precision

Los scores de confianza ayudan a:
- Priorizar el patron mas relevante
- Filtrar detecciones de baja calidad
- Proporcionar feedback sobre la certeza del analisis

---

**Siguiente notebook recomendado**: `pattern_evaluation.ipynb` para evaluar la precision del detector.